# Handling date and time variables
Rebuilt for pandas 3. Data: `data/orders.csv` (1,000 orders of an online shop: date, product, city, number of orders) and `data/messages.csv` (the send times of 1,000 chat messages; the message text is left out).

In [ ]:
# libraries for tables, numbers and charts
import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
# load both files; dates arrive as plain text (dtype str)
orders = pd.read_csv("data/orders.csv")
messages = pd.read_csv("data/messages.csv")
print(orders.dtypes)
orders.head()

## 1. Turn the text into dates

In [ ]:
# pd.to_datetime reads the text as dates; pandas 3 stores them as datetime64[us]
orders["date"] = pd.to_datetime(orders["date"])
orders.info()

In [ ]:
# the same in one step while reading the file (Note 15, parse_dates)
pd.read_csv("data/orders.csv", parse_dates=["date"])["date"].dtype

## 2. Parts of a date

In [ ]:
# year, month number, month name and day of the month
orders["date_year"] = orders["date"].dt.year
orders["date_month_no"] = orders["date"].dt.month
orders["date_month_name"] = orders["date"].dt.month_name()
orders["date_day"] = orders["date"].dt.day
orders[["date", "date_year", "date_month_no", "date_month_name", "date_day"]].head()

In [ ]:
# day of the week: Monday = 0 ... Sunday = 6, and its name
orders["date_dow"] = orders["date"].dt.dayofweek
orders["date_dow_name"] = orders["date"].dt.day_name()

# weekend flag: Saturday (5) and Sunday (6) give 1, other days 0
orders["date_is_weekend"] = (orders["date_dow"] >= 5).astype(int)
orders[["date", "date_dow", "date_dow_name", "date_is_weekend"]].head()

In [ ]:
# week of the year: .dt.week was removed in pandas 2, use isocalendar()
orders["date_week"] = orders["date"].dt.isocalendar().week

# quarter (1-4), and semester: quarters 1-2 are semester 1, quarters 3-4 semester 2
orders["quarter"] = orders["date"].dt.quarter
orders["semester"] = np.where(orders["quarter"].isin([1, 2]), 1, 2)
orders[["date", "date_week", "quarter", "semester"]].head()

In [ ]:
# ISO weeks can belong to the next year: 30 December 2019 is week 1 of 2020
pd.Series(pd.to_datetime(["2019-12-29", "2019-12-30"])).dt.isocalendar()

In [ ]:
# how many orders fall on each weekday (Monday first)
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
px.bar(orders["date_dow_name"].value_counts().reindex(days), title="Rows per weekday")

## 3. Time passed since a date
We fix "today" so that the numbers never change. `pd.Timestamp.now()` would give the moment the cell runs.

In [ ]:
# a fixed 'today': 21 April 2021, 16:02:16
today = pd.Timestamp("2021-04-21 16:02:16")

# subtracting gives a Timedelta for every row
elapsed = today - orders["date"]
elapsed.head()

In [ ]:
# whole days only
elapsed.dt.days.head()

In [ ]:
# days with the fraction of a day: divide by a one-day Timedelta
(elapsed / np.timedelta64(1, "D")).head()

In [ ]:
# np.timedelta64(1, 'M') (a month) now raises an error: a month has no fixed length
try:
    elapsed / np.timedelta64(1, "M")
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# months, way 1: average month = 365.2425 / 12 = 30.436875 days, then round
months_avg = (elapsed / pd.Timedelta(days=30.436875)).round()

# months, way 2: count calendar months from the year and month numbers
months_cal = (today.year - orders["date"].dt.year) * 12 + (today.month - orders["date"].dt.month)

pd.DataFrame({"date": orders["date"], "average_months": months_avg, "calendar_months": months_cal}).loc[[0, 1, 2, 3, 4, 11]]

In [ ]:
# the two ways disagree in this many of the 1,000 rows
(months_avg != months_cal).sum()

## 4. Parts of a time

In [ ]:
# the messages file: date and time together
messages["date"] = pd.to_datetime(messages["date"])

# hour, minute, second, and the time of day on its own
messages["hour"] = messages["date"].dt.hour
messages["min"] = messages["date"].dt.minute
messages["sec"] = messages["date"].dt.second
messages["time"] = messages["date"].dt.time
messages.head()

In [ ]:
# almost every message is sent between 22:00 and 03:00
px.bar(messages["hour"].value_counts().sort_index(), title="Messages per hour of the day")

In [ ]:
# time passed, in seconds, minutes and hours
# careful: 'm' is minutes, 'M' would be months
since = today - messages["date"]
pd.DataFrame({
    "elapsed": since,
    "seconds": since / np.timedelta64(1, "s"),
    "minutes": since / np.timedelta64(1, "m"),
    "hours": since / np.timedelta64(1, "h"),
}).head()

## 5. When pd.to_datetime has to guess the format

In [ ]:
# pandas guesses one format from the first value and uses it for every row
slash = pd.Series(["10/12/2019", "25/12/2019"])
try:
    pd.to_datetime(slash)
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

In [ ]:
# tell pandas the format (day first) instead
pd.to_datetime(slash, format="%d/%m/%Y")

In [ ]:
# values that cannot be read become NaT (not a time) with errors='coerce'
pd.to_datetime(pd.Series(["2019-12-10", "not a date"]), errors="coerce")